# 22 · End-to-end coherent transceiver

Every earlier notebook isolated one impairment and the block that removes
it.  Here they meet in one link, in the order a real system has them, and
the receiver runs the standard coherent DSP chain:

```
Tx: LDPC-coded bits -> frame (ZC preamble + 16-QAM payload, two polarizations) -> RRC -> transmitter bandwidth
Ch: 8 x 80 km fiber (CD, PMD, Kerr, ASE) -> LO: frequency offset, phase noise
Rx: unknown delay, IQ imbalance, receiver noise
DSP: IQ correction -> CDC -> FOE -> frame sync -> matched filter
     -> 2x2 LMS with inline BPS -> payload -> SNR, BER, GMI
     -> LLRs -> LDPC decoding -> post-FEC BER, FER
```

The receiver is given only what a real one knows: the frame layout, the
fiber length and its dispersion, the pulse.  It does not know the delay, the
frequency offset, the phase, the polarization state or the PMD.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

import commkit as ck
from commkit import RRC, Constellation, Preamble, SingleCarrierFrame
from commkit.backend import to_device
from commkit.coding import LDPCCode, decode
from commkit.frequency import MthPower
from commkit.impairments import Fiber, GramSchmidt
from commkit.recovery import BPS, CycleSlip

DEVICE = "gpu" if ck.backend.is_cupy_available() else "cpu"
print(DEVICE)

ck.plotting.apply_default_theme()
C16 = Constellation.qam(16)
RS, SPS = 32e9, 2
CODE = LDPCCode.dvbs2("5/6", "short")


def host(a):
    return to_device(a, "cpu")

## 1. Transmitter

A dual-polarization frame: a Zadoff-Chu preamble of 127 symbols (one root
per polarization) followed by 32400 16-QAM payload symbols per
polarization, RRC-shaped at 2 samples per symbol, then a 25 GHz transmitter
bandwidth (DAC and driver).  The payload carries eight codewords per
polarization of the short `"5/6"` LDPC code of DVB-S2: 16200 bits per
codeword, 13320 of them information, so its rate is 0.822, not 5/6 (see
[07](07-channel-coding.ipynb#Block-sizes:-codewords,-bits-and-symbols)).
Each codeword fills 16200 / 4 = 4050 16-QAM symbols, so eight are 32400;
the frame draws the information bits, encodes them and maps the codewords.

In [ ]:
frame = SingleCarrierFrame(
    payload_blocks=8,  # 8 codewords x 16200 bits / 4 bits per symbol = 32400 symbols
    payload_constellation=C16,
    payload_code=CODE,
    payload_rng=42,
    preamble=Preamble(sequence_type="zc", num_symbols=127, num_channels=2),
    num_channels=2,
)
tx = frame.to_signal(sps=SPS, symbol_rate=RS, pulse=RRC(rolloff=0.1), device=DEVICE)
tx = ck.filtering.iir_filter(
    tx,
    ck.filtering.bessel_sos(
        sampling_rate=tx.sampling_rate, cutoff=25e9, order=4, norm="mag"
    ),
)
# The IQ modulator puts the field on a 193.4 THz (1550.1 nm) carrier.
tx = tx.replace(domain="OPT", center_frequency=193.4e12)
print(tx)

## 2. Channel and receiver front end

* **Fiber**: 8 spans of 80 km of standard fiber with PMD
  (0.1 ps/√km) at a launch power of 2 dBm, an amplifier with a 5 dB noise
  figure after each span ([20](20-nonlinear-fiber-and-backpropagation.ipynb)).
* **LO**: a 1.2 GHz frequency offset and phase noise of 200 kHz combined
  linewidth, shared by both polarizations ([09](09-laser-phase-noise.ipynb), [12](12-frequency-offset-estimation.ipynb)).
* **Capture**: the frame starts 3000.3 samples into the record
  ([11](11-frame-synchronization-and-timing.ipynb)).
* **Receiver**: 1 dB / 5° IQ imbalance ([10](10-front-end-impairments.ipynb)) and thermal noise
  25 dB below the signal.

In [ ]:
fiber = Fiber(pmd_ps_sqrt_km=0.1)
LINK = dict(fiber=fiber, span_length_km=80.0, num_spans=8)
L_KM = LINK["span_length_km"] * LINK["num_spans"]

xp = tx.xp
silence = xp.zeros((2, 3000), dtype=tx.samples.dtype)
x = tx.replace(samples=xp.concatenate([silence, tx.samples, silence], axis=-1))
x = ck.impairments.apply_fiber_propagation(
    x, launch_power_dbm=2.0, noise_figure_db=5.0, steps_per_span=10, rng=1, **LINK
)
x = ck.spectral.shift_frequency(x, frequency=1.2e9)
x = ck.impairments.apply_phase_noise(x, linewidth=200e3, shared_lo=True, rng=2)
x = ck.timing.fft_fractional_delay(x, delay=0.3)
x = ck.impairments.apply_iq_imbalance(
    x, amplitude_imbalance_db=1.0, phase_imbalance_deg=5.0
)
rx = ck.impairments.apply_awgn(x, esn0_db=25, signal_power=1 / SPS, rng=3)

ck.plotting.plot_psd(
    rx, nperseg=2048, title="Received spectrum (both polarizations)", show=True
)

## 3. Receiver DSP, step by step

The order follows from what each block needs:

1. **IQ imbalance** first: it is a property of the receiver, applied last
   in the chain, so it is undone first.
2. **CDC**: the dispersion is known from the fiber; undoing it collapses
   the 8 x 12 = 100-symbol smear before anything that needs symbols.
3. **FOE** (blind M-th power): the frame correlation and the equalizer
   both need the frequency offset removed.
4. **Frame synchronization** on the ZC preamble, then the **matched
   filter**.
5. **2x2 LMS with inline BPS**, trained on the preamble and the first
   payload symbols: polarization demultiplexing, PMD, residual ISI and the
   carrier phase in one block.

In [ ]:
iq = ck.impairments.estimate_iq_imbalance(rx)
print(
    "IQ imbalance per polarization:",
    np.round(host(iq.amplitude_imbalance_db), 2),
    "dB,",
    np.round(host(iq.phase_imbalance_deg), 2),
    "deg (true 1.0 dB, 5 deg)",
)
y = ck.impairments.correct_iq_imbalance(rx, GramSchmidt())
y = ck.filtering.correct_chromatic_dispersion(
    y,
    dispersion_ps_nm_km=fiber.dispersion_ps_nm_km,
    fiber_length_km=L_KM,
)
foe = ck.frequency.estimate_frequency_offset(y, MthPower(), constellation=C16)
print(
    "frequency offset per polarization:",
    np.round(host(foe.value) / 1e9, 5),
    "GHz (true 1.2)",
)
y = ck.frequency.correct_frequency_offset(y, foe)

timing = ck.timing.estimate_timing(y)
print("frame start:", host(timing.value), "samples")
y = ck.timing.correct_timing(y, timing, mode="slice")
y = y.replace(samples=y.samples[..., : tx.samples.shape[-1]])
y = ck.filtering.matched_filter(y)

The frame is found about 7 samples later than the 3000.3 it was placed at.
That is not an error of the synchronizer but physics.  CDC ran while the
signal was still 1.2 GHz off centre, and dispersion turns a frequency shift
`Δf` into a delay `D·L·λ²Δf/c`: 17 ps/(nm·km) · 640 km · 0.0096 nm =
104 ps, 6.7 samples at 64 GS/s.  The transmitter's causal Bessel filter adds
its group delay, the remaining 0.4 samples.  The frame start found here is
where the frame really is in the processed record; a receiver that wants
the bulk delay small runs FOE before CDC.

Now the equalizer.  Its training sequence is the frame's own symbols
(preamble and body), which the receiver can rebuild from the frame
description.

In [ ]:
known = ck.backend.to_device(
    np.concatenate([frame.preamble.symbols, frame.body_symbols], axis=-1), DEVICE
)
TRAIN = 4000
res = ck.equalization.lms(
    y,
    num_taps=31,
    step_size=1e-3,
    constellation=C16,
    training_symbols=known[..., :TRAIN],
    cpr=BPS(cycle_slip=CycleSlip(history=100)),
)
ck.plotting.plot_equalizer_result(res, show=True)

symbols = y.replace(
    samples=res.y_hat, sampling_rate=RS
)  # one sample per symbol, the frame attached
payload = ck.core.extract_payload(symbols)

## 4. Results

The payload `Signal` carries its reference, so the metrics apply directly.
The noise variance for the GMI is the measured one (`10^(-SNR/10)`).

In [ ]:
skip = 2 * TRAIN
snr_db = ck.metrics.snr(payload, num_skip_symbols=skip)
print("SNR per polarization:", np.round(snr_db, 2), "dB")
print("BER per polarization:", ck.metrics.ber(payload, num_skip_symbols=skip))
noise_var = float(
    np.mean(ck.metrics.estimate_noise_variance(payload, num_skip_symbols=skip))
)
gmi = ck.metrics.gmi(payload, noise_var=noise_var, num_skip_symbols=skip)
print("GMI per polarization:", np.round(gmi, 3), "bits/symbol of 4")
print(
    f"information rate at the GMI: {RS * float(np.sum(gmi)) / 1e9:.0f} Gb/s over both polarizations"
)

fig, axes = plt.subplots(1, 2, figsize=(10, 4.5))
for p, ax in enumerate(axes):
    ck.plotting.plot_constellation(
        payload.samples[p, skip:],
        ax=ax,
        constellation=C16,
        title=f"Payload, polarization {'XY'[p]}",
    )
plt.show()

The LDPC decoder takes the LLRs of the received payload, computed with the
same measured noise variance, and returns the information bits.  The
payload's reference holds the information bits the frame encoded, so the
post-FEC error rates compare against them:

In [ ]:
llrs = ck.mapping.compute_llr(payload, noise_var=noise_var)
dec = decode(llrs, code=CODE)
print("pre-FEC BER (all symbols): ", ck.metrics.ber(payload))
print("post-FEC BER:", ck.metrics.ber(dec, payload))
print("post-FEC FER:", ck.metrics.fer(dec, payload))
print("decoder iterations per codeword:", host(dec.iterations).tolist())

Over all symbols, the equalizer's convergence included, the pre-FEC BER is
under 1e-3, and the decoder corrects every codeword in at most two
iterations; the first codeword, received while the equalizer was still
converging, takes the most.  At 19.4 dB the code has a wide margin: it
carries 4 · 13320/16200 = 3.29 information bits per symbol, far below the
GMI of 4, a net 210 Gb/s over both polarizations.

## 5. Where the SNR goes

A link budget explains the result.  The amplifier noise alone allows
`P_launch / P_ASE`, the Kerr effect adds nonlinear interference, the
receiver contributes its own noise, and the DSP leaves implementation
penalties (equalizer misadjustment, CPR noise).  Removing the impairments
one at a time shows each share:

In [ ]:
def chain_snr(*, nonlinear=True, receiver_noise=True):
    """SNR of the same link with some impairments switched off (ideal sync)."""
    z = tx.replace(samples=xp.concatenate([silence, tx.samples, silence], axis=-1))
    power = 2.0 if nonlinear else -30.0  # -30 dBm: Kerr effect negligible...
    z = ck.impairments.apply_fiber_propagation(
        z,
        launch_power_dbm=power,
        noise_figure_db=5.0 if nonlinear else None,
        steps_per_span=10,
        rng=1,
        **LINK,
    )
    if not nonlinear:  # ... and the ASE of the 2 dBm link added by hand
        z = ck.impairments.apply_awgn(z, esn0_db=ase_esn0, signal_power=1 / SPS, rng=4)
    if receiver_noise:
        z = ck.impairments.apply_awgn(z, esn0_db=25, signal_power=1 / SPS, rng=3)
    z = ck.filtering.correct_chromatic_dispersion(
        z,
        dispersion_ps_nm_km=fiber.dispersion_ps_nm_km,
        fiber_length_km=L_KM,
    )
    z = ck.filtering.matched_filter(
        z.replace(samples=z.samples[..., 3000 : 3000 + tx.samples.shape[-1]])
    )
    r = ck.equalization.lms(
        z,
        num_taps=31,
        step_size=1e-3,
        constellation=C16,
        training_symbols=known[..., :TRAIN],
    )
    return float(
        np.mean(
            ck.metrics.snr(
                ck.core.extract_payload(z.replace(samples=r.y_hat, sampling_rate=RS)),
                num_skip_symbols=skip,
            )
        )
    )


# ASE-only SNR of the link: noise of 8 amplifiers in the signal band.
h_nu = 6.62607015e-34 * tx.center_frequency
gain = 10 ** (0.2 * 80 / 10)
p_ase = (
    8 * (10**0.5 * gain - 1) * h_nu / 2 * RS * 2
)  # W, both polarizations, in the symbol-rate band
ase_esn0 = 10 * np.log10(10 ** (2.0 / 10) * 1e-3 / p_ase)
print(f"ASE-limited SNR (theory):              {ase_esn0:5.2f} dB")
print(
    f"ASE only, through the DSP:             {chain_snr(nonlinear=False, receiver_noise=False):5.2f} dB"
)
print(
    f"ASE + Kerr (the fiber model):          {chain_snr(receiver_noise=False):5.2f} dB"
)
print(f"ASE + Kerr + receiver noise:           {chain_snr():5.2f} dB")
print(f"full chain with lasers and front end:  {float(np.mean(snr_db)):5.2f} dB")

The amplifier noise sets the ceiling, the DSP loses about half a dB of it,
the Kerr effect at 2 dBm costs about 5 dB and the receiver noise another
1.5 dB.  The frequency offset, the phase noise, the IQ imbalance and the
unknown delay cost nothing measurable: the last two rows differ only by
their noise realizations.

**Summary**

* The DSP chain undoes the impairments in reverse order of how much they
  need from each other: receiver front end, dispersion, frequency, timing,
  then the adaptive 2x2 equalizer with carrier recovery.
* Each block is one function call on the `Signal`; the frame and the
  reference travel with the data until the metrics, and the reference's
  information bits until the decoder's output.
* The link budget, stage by stage, shows where the SNR is lost.

This closes the receiver part of the series.  Notebooks
[23](23-carrier-phase-characterization.ipynb)-[28](28-measurement-template-self-homodyne-iq.ipynb) turn to measurement: characterizing the
lasers whose phase noise the receiver had to track.